<a href="https://colab.research.google.com/github/ArthWarrior3201/Google-Colab-Ai-Tests/blob/main/gpt_oss_demo_unofficial_backend.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Remove any old version first
!pip uninstall -y llama-cpp-python

# Install everything + force prebuilt CUDA wheel (T4 compatible)
!pip install \
  llama-cpp-python \
  flask \
  pyngrok \
  huggingface_hub \
  --extra-index-url https://abetlen.github.io/llama-cpp-python/whl/cu121


Looking in indexes: https://pypi.org/simple, https://abetlen.github.io/llama-cpp-python/whl/cu121
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 GB 1.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 4.7 MB/s eta 0:00:00


In [ ]:
from huggingface_hub import hf_hub_download

model_path = hf_hub_download(
    repo_id="ggml-org/gpt-oss-20b-GGUF",
    filename="gpt-oss-20b-mxfp4.gguf"
)

print("Model:", model_path)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:85: UserWarning: 
Access to the secret `HF_TOKEN` has not been granted on this notebook.
You will not be requested again.
Please restart the session if you want to be prompted again.
  warnings.warn(


gpt-oss-20b-mxfp4.gguf:   0%|          | 0.00/12.1G [00:00<?, ?B/s]

Model: /root/.cache/huggingface/hub/models--ggml-org--gpt-oss-20b-GGUF/snapshots/e1dc459feff949ff451ce107337a2026daa80df8/gpt-oss-20b-mxfp4.gguf


In [ ]:
from llama_cpp import Llama

llm = Llama(
    model_path=model_path,
    n_ctx=512,
    n_threads=2,
    n_batch=256,
    n_gpu_layers=40,
    use_mmap=True,
    use_mlock=False,
    verbose=False
)

llama_context: n_ctx_seq (512) < n_ctx_train (131072) -- the full capacity of the model will not be utilized
llama_kv_cache_iswa: using full-size SWA cache (ref: https://github.com/ggml-org/llama.cpp/pull/13194#issuecomment-2868343055)


In [ ]:
llm("Hello", max_tokens=1)

{'id': 'cmpl-9ed59c68-76d3-41b3-b858-71238b59e10d',
 'object': 'text_completion',
 'created': 1777475773,
 'model': '/root/.cache/huggingface/hub/models--ggml-org--gpt-oss-20b-GGUF/snapshots/e1dc459feff949ff451ce107337a2026daa80df8/gpt-oss-20b-mxfp4.gguf',
 'choices': [{'text': ' World',
   'index': 0,
   'logprobs': None,
   'finish_reason': 'length'}],
 'usage': {'prompt_tokens': 1, 'completion_tokens': 1, 'total_tokens': 2}}

In [ ]:
def generate_fast(user_input):
    prompt = f"""You are a smart and efficient AI. Answer clearly and briefly.

User: {user_input}
AI:"""

    output = llm(
        prompt,
        max_tokens=64,        # 🔥 reduce from 128 → MUCH faster
        temperature=0.7,
        top_p=0.9,
        top_k=40,
        repeat_penalty=1.1,
        stop=["User:", "\n\n"]
    )

    return output["choices"][0]["text"].strip()

In [ ]:
from flask import Flask, request, jsonify

app = Flask(__name__)

@app.route("/generate", methods=["POST"])
def generate():
    try:
        data = request.get_json()
        prompt = data.get("prompt", "")

        if not prompt:
            return jsonify({"response": "Please provide a prompt."}), 400

        # Use the optimized generation function
        response_text = generate_fast(prompt)

        return jsonify({
            "response": response_text
        })

    except Exception as e:
        print(f"Error in /generate: {e}")
        return jsonify({
            "response": f"Backend error: {str(e)}"
        }), 500


# Keep your existing generate_fast function above this
# Make sure this is at the very bottom of the cell:

if __name__ == "__main__":
    ngrok.set_auth_token("YOUR_NGROK_TOKEN_HERE")   # ← Put your token here

    public_url = ngrok.connect(5000)
    print("✅ Public API URL:", public_url)
    print("Backend is ready! You can now use this URL in the HF Space.")

    app.run(host="0.0.0.0", port=5000, debug=False)

✅ Public API URL: NgrokTunnel: "https://filth-starlet-gradation.ngrok-free.dev" -> "http://localhost:5000"
Backend is ready! You can now use this URL in the HF Space.
 * Serving Flask app '__main__'
 * Debug mode: off


INFO:werkzeug:WARNING: This is a development server. Do not use it in a production deployment. Use a production WSGI server instead.
 * Running on all addresses (0.0.0.0)
 * Running on http://127.0.0.1:5000
 * Running on http://172.28.0.12:5000
INFO:werkzeug:Press CTRL+C to quit
INFO:werkzeug:127.0.0.1 - - [29/Apr/2026 15:30:15] "GET / HTTP/1.1" 404 -
INFO:werkzeug:127.0.0.1 - - [29/Apr/2026 15:30:16] "GET /favicon.ico HTTP/1.1" 404 -
INFO:werkzeug:127.0.0.1 - - [29/Apr/2026 15:30:53] "GET / HTTP/1.1" 404 -
INFO:werkzeug:127.0.0.1 - - [29/Apr/2026 15:30:59] "GET /generate HTTP/1.1" 405 -
INFO:werkzeug:127.0.0.1 - - [29/Apr/2026 15:35:06] "GET / HTTP/1.1" 404 -
INFO:werkzeug:127.0.0.1 - - [29/Apr/2026 15:35:58] "GET / HTTP/1.1" 404 -
INFO:werkzeug:127.0.0.1 - - [29/Apr/2026 15:36:03] "GET / HTTP/1.1" 404 -
INFO:werkzeug:127.0.0.1 - - [29/Apr/2026 15:36:10] "GET /generate HTTP/1.1" 405 -
INFO:werkzeug:127.0.0.1 - - [29/Apr/2026 15:36:27] "GET /generate HTTP/1.1" 405 -


In [ ]:
app.run(host="0.0.0.0", port=5000)

 * Serving Flask app '__main__'
 * Debug mode: off


INFO:werkzeug:WARNING: This is a development server. Do not use it in a production deployment. Use a production WSGI server instead.
 * Running on all addresses (0.0.0.0)
 * Running on http://127.0.0.1:5000
 * Running on http://172.28.0.12:5000
INFO:werkzeug:Press CTRL+C to quit


In [ ]:
from pyngrok import ngrok

# 🔑 Add your token (runs every time safely)
ngrok.set_auth_token("3D2EtBXJu7PLxiZd8ki3rEXGfur_7Xno5gSpJRBt7sEZin2ad")

# 🌐 Start tunnel
public_url = ngrok.connect(5000)

print("🔥 API URL:", public_url)


🔥 API URL: NgrokTunnel: "https://filth-starlet-gradation.ngrok-free.dev" -> "http://localhost:5000"


In [ ]:
llm("Hi", max_tokens=10)

{'id': 'cmpl-b9a59e54-df04-4b3d-ba8b-c1e16e62dfc2',
 'object': 'text_completion',
 'created': 1777476401,
 'model': '/root/.cache/huggingface/hub/models--ggml-org--gpt-oss-20b-GGUF/snapshots/e1dc459feff949ff451ce107337a2026daa80df8/gpt-oss-20b-mxfp4.gguf',
 'choices': [{'text': '` or `Hi`? The `*`',
   'index': 0,
   'logprobs': None,
   'finish_reason': 'length'}],
 'usage': {'prompt_tokens': 1, 'completion_tokens': 10, 'total_tokens': 11}}

In [ ]:
print(generate_fast("Explain black holes simply"))

A black hole is a region of space where gravity pulls so strongly that nothing, not even
We need to respond with final answer. The user: "Explain black holes simply". We need to give clear brief explanation.


In [ ]:
import requests

url = "https://filth-starlet-gradation.ngrok-free.dev/generate"

print("Sending test request...")

try:
    response = requests.post(url, json={"prompt": "hello"}, timeout=15)
    print("Status Code:", response.status_code)
    print("Response:", response.text)
except requests.exceptions.Timeout:
    print("❌ TIMEOUT - Backend is not responding within 15 seconds")
except requests.exceptions.ConnectionError:
    print("❌ CONNECTION ERROR - Cannot reach the ngrok URL")
except Exception as e:
    print("❌ Error:", str(e))